# Options 1

In [ ]:
%run cleaning.ipynb import df_clean_text as df
import re
from modules.stopword import stop_words
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from collections import Counter
from modules.keyword_mapping import keyword_mapping

In [ ]:
if len(df) < 1:
    raise ValueError(" Pas assez de données pour effectuer un clustering.")
"""converts texts into matrix of TF-IDF features, which is a common technique in natural language processing to represent text data numerically."""
vectorizer = TfidfVectorizer(
    max_features= 50,
    min_df = 2,
    max_df = 0.8, 
    stop_words= stop_words,           
    ngram_range=(2, 4),
    lowercase=True
)

X = vectorizer.fit_transform(df['title_clean'])
X_document_name = vectorizer.get_feature_names_out(df['title_clean'])
row, col = X.nonzero()
document_name = X.data
document_name = [X_document_name[i] for i in col]
document_name = [f"{name} ({count})" for name, count in zip(document_name, X.data)] 
# print(document_name)


def find_optimal_clusters(X, max_k = 44):
# the maximum number of cluster should not exceed the half of the documents length
    max_possible_k = min(max_k, len(df) // 2)
    if max_possible_k < 2:
         return 2
# Test for each number of cluster which gives the best score
    k_range = range(2, max_possible_k)
# Stock each cluster and its corresponding score inside the "scores" table 
    scores = []
    for k in k_range:
        try:
            kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
            kmeans.fit(X)
            score = silhouette_score(X, kmeans.labels_)
            scores.append(score)
        except:
            scores.append(-1)  #  Clustering errors

    if not scores or all(s == -1 for s in scores):
        return 2

    best_k = k_range [scores.index(max(scores))]
    print(f"best_k = {best_k}, score = {score}")
    return best_k
    
n_clusters = find_optimal_clusters(X) 
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(X)

df_optimal = df['cluster'].value_counts().reset_index()
df_optimal.columns = ['cluster', 'count']


def label_cluster(df, cluster_id):
    # assign a cluster for each document
    titles = df[df['cluster'] == cluster_id]['title_clean']
    words = []
    # print(f"Cluster {cluster_id} - Titles: {len(titles)} : \n {titles}") # Debug: nombre de titres dans le cluster
    for word in titles:
        titles_parts = word.split()
        words.extend(titles_parts) # Unigrammes
        words.extend([f"{titles_parts[i]} {titles_parts[i+1]}" for i in range(len(titles_parts)-1)]) # Bigrammes

    frequency = Counter(words)                              
    top_terms = [w for w, c in frequency.most_common(10)] # Nouveau: 10 termes
    text = " ".join(top_terms).lower()

    scores = {}
    for label, keywords in keyword_mapping.items():
        score = sum(1 for k in keywords if re.search(r'\b' + re.escape(k) + r'\b', text)) # Nouveau: correspondance exacte
         # Compte les correspondances
        if score > 0:
            scores[label] = score
            
    # Si des correspondances sont trouvées, choisir celle avec le score le plus élevé
    if scores:
        best_label = max(scores, key=scores.get)
        # print(text, best_label, scores[best_label])
        return best_label # Utiliser .title() si vous voulez des majuscules
    
    return top_terms[0] if top_terms else f"Groupe {cluster_id}"

cluster_labels = {cid: label_cluster(df, cid) for cid in range(n_clusters)}
df['famille_poste'] = df['cluster'].map(cluster_labels)
for category in df['famille_poste'].unique():
    print(f"Cluster: {df[df['famille_poste'] == category]['cluster'].iloc[0]} | Category: {category} | Number of titles: {len(df[df['famille_poste'] == category])}")
    print(df[df['famille_poste'] == category]['title_clean'].head(5))  # Affiche les 5 premiers titres de chaque catégorie

In [ ]:
print(df.keys())